# Compute Forward Operator

## 📦 Imports

In [1]:
import mindmap as mm 
import mne
import pyvista as pv

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- `bem`: Boundary Element Model (Head conductor model).
- `src-raw`: Source space (before computing forward).
- `eeg`: EEG.
- `trans`: Affine transformation for EEG coregistration.

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)
- `hmodel`: Type of head model ["bem", "bem-sim"]. Whether to compute the forward solution for inverse problems or simulations.

In [2]:
# ------------ Parameters -------------
fconfig = "../../papers/simf/configuration.yml"
sub     = 1
hmodel  = "bem"
# -------------------------------------

reader = mm.io.dataset_reader(fconfig)
mm_dir = reader.config["MM_DIR"]

# data
data = reader(sub, hmodel, "src-raw", "eeg", "trans")

DATA SUMMARY
Dataset: simf
Subject: vepcon_sub-10
Retrieved objects:
- bem
- trans
- eeg
- src-raw
Missing objects:


📊 **Check coregistration**

In [3]:
head = data[hmodel]["surfs"][0]
sens = data["eeg"]._get_channel_positions()
sens = mne.transforms.apply_trans(data["trans"], sens)

# plot
head = mm.head.surf_to_polydata(head["rr"], head["tris"])
sens = pv.PolyData(sens)

p = pv.Plotter()
p.add_mesh(head, color="#D2A18C", smooth_shading=False, label="head")
p.add_mesh(sens, render_points_as_spheres=True, color="black", label="sensors", point_size=8)

p.add_legend(face="circle")
p.show()

Widget(value='<iframe src="http://localhost:42391/index.html?ui=P_0x7742dc8f5490_0&reconnect=auto" class="pyvi…

## ⚙️ Compute Forward Solution

In [4]:
forward = mne.make_forward_solution(
    info    = data["eeg"].info,
    trans   = data["trans"],
    src     = data["src-raw"],
    bem     = data[hmodel],
    meg     = False,
    eeg     = True,
    verbose = "warning",
)

📊 **Plot leadfields**

In [5]:
mm.viz.plot_leadfield(data["eeg"], forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:42391/index.html?ui=P_0x7742d9c921d0_1&reconnect=auto" class="pyvi…

💾 **Save Forward Solution** <br>
🎚️ **<font color=pink>Save</font>**:
- `overwrite`: Overwrite existing forward solution.


In [6]:
# ------------ Parameters -------------
overwrite = True
# -------------------------------------

if hmodel == "bem":
    ffwd = mm_dir/data["sub"]/"fwd"/f"{data['sub']}_desc-inverse_fwd.fif"
else:
    ffwd = mm_dir/data["sub"]/"fwd"/f"{data['sub']}_desc-simulation_fwd.fif"

if not ffwd.parent.exists():
    ffwd.parent.mkdir(parents=True)

mne.write_forward_solution(ffwd, forward, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved forward solution for subject {data['sub']} with model '{hmodel}'\033[0m")

Successfully saved forward solution for subject vepcon_sub-10 with model 'bem'
